# Phase 2 - EDA + Preprocess (SME Early-Warning)
Upload `data.csv` from `Downloads/archive (3)` when prompted. Exports preprocessor + final 22-feature list. No median-fill at inference - see docs/PHASE2_FINDINGS.md.

In [ ]:
import pandas as pd, numpy as np, json
from google.colab import files
up = files.upload()  # upload data.csv
df = pd.read_csv('data.csv')
df.columns = [c.strip() for c in df.columns]
print(df.shape, df['Bankrupt?'].value_counts().to_dict(), df.isna().sum().sum())

In [ ]:
# EDA: imbalance + duplicates
print('pos rate %.2f%%' % (df['Bankrupt?'].mean()*100))
num = df.drop(columns=['Bankrupt?'])
c = num.corr().abs()
dups = [(c.index[i], c.columns[j], round(float(c.iloc[i,j]),3)) for i in range(len(c)) for j in range(i+1,len(c)) if c.iloc[i,j] > 0.88]
print('pairs>0.88:', len(dups))
for a,b,v in dups[:15]: print(a,'<->',b,v)

In [ ]:
# Preprocess: stratified split + RobustScaler fit on train only
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import RobustScaler
import pickle
X = df.drop(columns=['Bankrupt?']); y = df['Bankrupt?']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
sc = RobustScaler().fit(Xtr)
print('train', Xtr.shape, 'test', Xte.shape)
FINAL22 = ['Continuous interest rate (after tax)','Borrowing dependency','Persistent EPS in the Last Four Seasons','Total income/Total expense','Total debt/Total net worth','Net profit before tax/Paid-in capital','Debt ratio %','Interest Expense Ratio','Degree of Financial Leverage (DFL)','Net Value Per Share (A)','Retained Earnings to Total Assets','Contingent liabilities/Net worth','Operating Profit Rate','ROA(A) before interest and % after tax','Inventory and accounts receivable/Net value','Non-industry income and expenditure/revenue','Accounts Receivable Turnover','Quick Ratio','Current Liability to Assets','Working Capital to Total Assets','Cash Flow to Total Assets','Interest Coverage Ratio (Interest expense to EBIT)']
missing = [f for f in FINAL22 if f not in X.columns]
print('missing from CSV:', missing)
json.dump(FINAL22, open('final22_features.json','w'), indent=1)
pickle.dump(sc, open('robust_scaler.pkl','wb'))
Xtr[FINAL22].to_csv('X_train_final22.csv', index=False)
print('saved final22_features.json, robust_scaler.pkl, X_train_final22.csv')
files.download('final22_features.json')